# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install transformers==4.48.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 29.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into

In [3]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [4]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [5]:
import requests  # For sending HTTP requests to the SerpApi

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [6]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...
[1] The best new destinations for your travel bucket list [2026]
    Five destinations on my travel bucket list · Table of Contents · Slovenia · Albania · Costa Rica · Iceland · Uzbekistan · Post navigation · Where to wander next ...
    Source: https://wanderwithlaura.com/destinations-for-your-travel-bucket-list/

[2] My Top 26 Best Travel Destinations For 2026
    My Top 26 Best Travel Destinations For 2026 · Kyrgyzstan · Kazakhstan · Uzbekistan · Tajikistan · Pakistan · Nepal · Mexico · Colombia. Colombia ...
    Source: https://thepartyingtraveler.com/2022/01/03/26-best-travel-destinations-to-visit-in-2026/

[3] Best Travel Destinations 2026
    Our team of travel experts have selected 25 must-visit destinations and 25 unforgettable experiences. And this year, you can book trips inspired by the list.
    Source: https://www.lonelyplanet.com/best-in-travel

[4] 30 World's Best Places to Visit for 2026
    World's Best Places to Visit for

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [7]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [8]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        # Query the GPT-4 model with the router prompt and user input
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [9]:
route_query("what is the revenue of uber in 2021?")


{'action': '10K_DOCUMENT_QUERY',
 'reason': "Requests Uber's annual financial revenue",
 'answer': "Uber's 2021 revenue"}

In [10]:
route_query("what is an AI Agent?")

{'action': 'OPENAI_QUERY',
 'reason': 'AI Agents are covered in OpenAI documentation',
 'answer': 'Software performing tasks autonomously'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [11]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [12]:
# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    !git clone https://github.com/hamzafarooq/multi-agent-course.git

Cloning into 'multi-agent-course'...
remote: Enumerating objects: 2823, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 2823 (delta 255), reused 265 (delta 223), pack-reused 2389 (from 2)
Receiving objects: 100% (2823/2823), 124.00 MiB | 18.01 MiB/s, done.
Resolving deltas: 100% (1069/1069), done.


In [13]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /content/multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [14]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/547M [00:00<?, ?B/s]

[ 1.2799683   0.4015841  -3.5162659  -0.39813185  1.5919138 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [15]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [15]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [16]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [17]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Uber's annual financial revenue
⚙️ Processing query...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



In [18]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for Lyft's annual revenue from a specific reporting year
⚙️ Processing query...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



In [19]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs
⚙️ Processing query...

Getting your response from the internet 🌐 ...
🤖 BOT RESPONSE:

[1] How did LLMs become the main AI model as opposed to ...
    I'm not technical by any means and this is probably a stupid question. But I just wanted to know how LLMs came to be the main AI model as its my understanding ...
    Source: https://www.reddit.com/r/ArtificialInteligence/comments/1m64o83/how_did_llms_become_the_main_ai_model_as_opposed/

[2] The LLM Critics Are Right. I Use LLMs Anyway.
    So this article is me trying to describe it. I'll start by going through all of the fair and valid concerns about using LLMs, the things that ...
    Source: https://www.theocharis.dev/blog/llm-critics-are-right-i-use-llms-anyway/

[3] Vector Search with LLMs - Computerphile
    This video was filmed and edited by Sean Riley. Vector Search with LLMs - Computer

In [20]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?

📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API feature
⚙️ Processing query...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



In [21]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents is covered by OpenAI Agents documentation
⚙️ Processing query...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [22]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'OPENAI_QUERY', 'INTERNET_QUERY'}
bob    (finance_analyst) → {'OPENAI_QUERY', '10K_DOCUMENT_QUERY'}
carol  (unknown user)    → set()


In [23]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [24]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks for a company's annual financial data

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [25]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Asks about Uber's annual financial revenue

✅ Access granted — processing...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



"RAG response error: name 'rag_formatted_response' is not defined"

In [26]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025

📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [27]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents

📍 Selected Route: OPENAI_QUERY
📝 Reason: Agent building is covered by OpenAI documentation

✅ Access granted — processing...

🤖 BOT RESPONSE:

RAG response error: name 'rag_formatted_response' is not defined



"RAG response error: name 'rag_formatted_response' is not defined"

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [28]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": sub_queries_prompt},
        ]
    )
  return response.choices[0].message.content


In [29]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

{"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}


In [30]:
# ── Part 1: implementation ──────────────────────────────────────────────

def _parse_sub_queries(user_query: str, raw: str) -> list:
    """
    Defensively parse sub_queries()'s output into a clean list[str].
    Expected shape: {"subQuestions": ["...", "..."]}, possibly wrapped
    in markdown fences or with stray text around the JSON.
    Falls back to [user_query] if parsing fails for any reason.
    """
    if not raw:
        return [user_query]

    try:
        json_match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not json_match:
            return [user_query]

        parsed = json.loads(json_match.group())
        sub_qs = parsed.get("subQuestions")

        if isinstance(sub_qs, list) and sub_qs:
            cleaned = [str(q).strip() for q in sub_qs if str(q).strip()]
            return cleaned if cleaned else [user_query]

        return [user_query]

    except (json.JSONDecodeError, AttributeError, TypeError):
        return [user_query]


def _answer_sub_query(sub_query: str, GREY: str, RESET: str) -> dict:
    """
    Route one sub-query through route_query + routes, mirroring the
    try/except structure of agentic_rag's Step 2 / Step 4 exactly.
    """
    entry = {"query": sub_query, "action": None, "reason": None, "result": None}

    try:
        response = route_query(sub_query)
    except Exception as route_err:
        entry["result"] = f"Routing error: {route_err}"
        return entry

    action = response.get("action")
    reason = response.get("reason")
    entry["action"] = action
    entry["reason"] = reason

    print(f"{GREY}   📍 Route: {action} — {reason}{RESET}")

    try:
        route_function = routes.get(action)
        if route_function:
            if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                entry["result"] = asyncio.run(route_function(sub_query, action))
            else:
                entry["result"] = route_function(sub_query, action)
        else:
            entry["result"] = f"Unsupported action: {action}"
    except Exception as exec_err:
        entry["result"] = f"Execution error: {exec_err}"

    return entry


def _synthesize_final_answer(user_query: str, sub_results: list) -> str:
    """
    Ask the model to merge the sub-answers into one coherent response,
    preserving any sources/citations that appear in them (e.g. the
    "[1] ... Source: ..." lines get_internet_content returns).
    Falls back to a labeled concatenation if the LLM call fails.
    """
    parts = [
        f'Sub-question: "{item["query"]}"\n'
        f'Route: {item["action"]}\n'
        f'Answer: {item["result"]}'
        for item in sub_results
    ]
    joined = "\n\n".join(parts)

    synth_prompt = f"""
    You are combining answers to sub-questions into one final answer to the
    user's original question. Write a single, well-organized response that
    addresses every sub-question. Preserve any citations or "Source:" links
    exactly as they appear in the sub-answers — never invent or drop them.
    Do not mention "sub-questions" or routing — just answer naturally.

    Original question: "{user_query}"

    Sub-question answers:
    {joined}

    Final answer:
    """

    try:
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": synth_prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception:
        # Graceful fallback: still readable, still keeps citations
        return joined


def agentic_rag_multi(user_query: str):
    """
    Split a compound query, run each sub-query through the agentic pipeline,
    and synthesise one final answer.

    Args:
        user_query (str): Possibly compound question.

    Returns:
        str: A single composed answer covering every sub-question.
    """
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # Step 1: Split into sub-queries, parsing defensively
    try:
        raw = sub_queries(user_query)
    except Exception as split_err:
        print(f"{GREY}⚠️ sub_queries() failed ({split_err}); treating as a single query.{RESET}\n")
        raw = None

    sub_qs = _parse_sub_queries(user_query, raw) if raw is not None else [user_query]

    print(f"{GREY}🔎 Sub-queries ({len(sub_qs)}):{RESET}")
    for i, q in enumerate(sub_qs, start=1):
        print(f"{GREY}  {i}. {q}{RESET}")
    print()

    # Step 2: Route + answer each sub-query
    sub_results = []
    for i, sq in enumerate(sub_qs, start=1):
        print(f"{GREY}⚙️ [{i}/{len(sub_qs)}] {sq}{RESET}")
        sub_results.append(_answer_sub_query(sq, GREY, RESET))
        print()

    # Step 3: Compose one final answer, keeping citations
    final_answer = _synthesize_final_answer(user_query, sub_results)

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{final_answer}\n")

    return final_answer


# Test cases from the table above
# agentic_rag_multi("what was uber revenue in 2021?")
# agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")
# agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

In [31]:
agentic_rag_multi("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

🔎 Sub-queries (1):
  1. What was Uber's revenue in 2021?

⚙️ [1/1] What was Uber's revenue in 2021?
   📍 Route: 10K_DOCUMENT_QUERY — Query asks for financial information from Uber's annual report

🤖 BOT RESPONSE:

Uber reported **$17.455 billion in revenue in 2021**.



'Uber reported **$17.455 billion in revenue in 2021**.'

In [32]:
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021

🔎 Sub-queries (2):
  1. What was Lyft's revenue in 2021?
  2. What was Uber's revenue in 2021?

⚙️ [1/2] What was Lyft's revenue in 2021?
   📍 Route: 10K_DOCUMENT_QUERY — Lyft's revenue is reported in its annual financial filings

⚙️ [2/2] What was Uber's revenue in 2021?
   📍 Route: 10K_DOCUMENT_QUERY — Asks about Uber's annual financial revenue

🤖 BOT RESPONSE:

- **Lyft revenue in 2021:** **$3.2 billion** (approximately **$3.218 billion**)
- **Uber revenue in 2021:** **$17.5 billion** (approximately **$17.455 billion**)



'- **Lyft revenue in 2021:** **$3.2 billion** (approximately **$3.218 billion**)\n- **Uber revenue in 2021:** **$17.5 billion** (approximately **$17.455 billion**)'

In [33]:
agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

👤 User Query: what was uber's 2021 revenue and what are the newest LLMs?

🔎 Sub-queries (2):
  1. What was Uber's revenue in 2021?
  2. What are the newest large language models (LLMs)?

⚙️ [1/2] What was Uber's revenue in 2021?
   📍 Route: 10K_DOCUMENT_QUERY — Asks for Uber's annual financial data

⚙️ [2/2] What are the newest large language models (LLMs)?
   📍 Route: INTERNET_QUERY — Requires current information about models across providers
Getting your response from the internet 🌐 ...

🤖 BOT RESPONSE:

- **Uber’s 2021 revenue:** **$17.455 billion**.

- **Newest large language models (LLMs):** The latest-model landscape changes quickly, so there is no single definitive “newest” model. The provided sources identify recent or notable models including **Nova Micro, Nova Lite, Nova Pro, Deep Think, and Pro**. They also discuss **GPT-4o**, OpenAI’s multimodal flagship released in May 2024, though it should not necessarily be considered the newest model by September 2026.

LLMs are AI sys

'- **Uber’s 2021 revenue:** **$17.455 billion**.\n\n- **Newest large language models (LLMs):** The latest-model landscape changes quickly, so there is no single definitive “newest” model. The provided sources identify recent or notable models including **Nova Micro, Nova Lite, Nova Pro, Deep Think, and Pro**. They also discuss **GPT-4o**, OpenAI’s multimodal flagship released in May 2024, though it should not necessarily be considered the newest model by September 2026.\n\nLLMs are AI systems that process large amounts of text to understand and generate human language.\n\nSources:  \nhttps://www.ibm.com/think/topics/large-language-models  \nhttps://en.wikipedia.org/wiki/List_of_large_language_models  \nhttps://llm-stats.com/llm-updates  \nhttps://botpress.com/blog/best-large-language-models  \nhttps://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-are-large-language-models-llms'

---

## Bonus (optional) — RBAC with a semantic cache

*Not graded on its own — but this is rehearsal for graded work. **ARGUS**, the full-stack
assignment for this module, requires a real caching layer and a cost panel showing spend per
100 sources with and without it. Build the cache here, on a system you already understand,
and you'll be extending it in ARGUS rather than meeting it for the first time under a deadline.*

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


In [34]:
# ── Bonus: your implementation ───────────────────────────────────────────────
# Reuse USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS from Section 6.

import numpy as np


def _cosine_distance(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 1.0
    return 1.0 - float(np.dot(a, b) / denom)


def is_time_sensitive(question: str) -> bool:
    """
    Heuristic borrowed from 003. Agentic Router_semantic_caching_rbac.ipynb:
    never cache anything whose correct answer depends on "right now" —
    a stock price cached an hour ago is a wrong answer served fast.
    """
    keywords = [
        "today", "tonight", "this week", "this month", "current", "currently",
        "now", "latest", "recent", "recently", "stock price", "live",
        "real-time", "real time", "as of today", "breaking",
    ]
    q = question.lower()
    return any(k in q for k in keywords)


class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice: TAGGED, partitioned by ACTION (route), not by user or role.

    Why: RBAC here is defined at the level of the action (OPENAI_QUERY /
    10K_DOCUMENT_QUERY / INTERNET_QUERY) — has_access() asks "can this role
    reach this source", never "can this role reach this other user's cache
    rows". So the cache is bucketed the same way permission is checked: one
    bucket per action. A lookup only ever searches the bucket for the action
    the router just chose, so an entry produced under 10K_DOCUMENT_QUERY can
    never surface for an INTERNET_QUERY lookup.

    What actually stops the leak is ordering, not the cache's internals:
    secure_agentic_rag_cached() only calls cache.check()/cache.add() *after*
    has_access(user_id, action) has already passed. A role that can't reach
    an action is turned away at the RBAC gate before the cache is touched —
    so bucketing by action (shared across roles) is safe, and it also fixes
    the waste the assignment calls out: engineer and finance_analyst both
    reach OPENAI_QUERY, so under this design that answer is computed once
    and reused by both, instead of once per role.

    As defense-in-depth (not the primary guard — see above), check()/add()
    also re-verify has_access(user_id, action) themselves before honouring a
    hit or storing a row, so a future caller that forgets the gate still
    can't leak or poison another role's data.

    On role changes: nothing needs to be invalidated. Permission is re-read
    from USERS/ROLE_PERMISSIONS on every call rather than cached, so the
    instant a role loses access to an action the gate denies before the
    cache is reached, and the instant a role gains access it immediately
    benefits from whatever other permitted roles already put in that
    action's bucket — there's no stale state to clean up.

    (A strictly per-role or per-user partition is the other viable design —
    also leak-safe, since the RBAC gate runs first either way — but it
    needlessly recomputes and re-stores identical answers per role/user.)
    """

    def __init__(self, threshold: float = 0.2):
        # threshold = max cosine *distance* (1 - cosine similarity) for a hit;
        # 0.2 distance ≈ similarity >= 0.8 required to count as a match.
        self.threshold = threshold
        # One list of entries per action, e.g. "10K_DOCUMENT_QUERY".
        # Each entry: {"question": str, "answer": str, "embedding": np.ndarray}
        self._buckets: dict = {}

    def _bucket(self, action: str) -> list:
        return self._buckets.setdefault(action, [])

    # NOTE: extended beyond the stub's (user_id, question) signature to also
    # take `action` — the route the query was just classified into. That's
    # what determines which bucket is safe to search (see class docstring);
    # re-deriving it inside the cache would mean a second, redundant router
    # call. `secure_agentic_rag_cached` always has `action` in hand already.
    def check(self, user_id: str, question: str, action: str):
        """Return (hit: bool, answer: str | None, embedding, similarity: float | None)."""
        embedding = get_text_embeddings(question)

        # Defense-in-depth: never honour a hit the caller isn't allowed to see,
        # even if this method is ever called before the RBAC gate by mistake.
        if not has_access(user_id, action):
            return False, None, embedding, None

        best_answer, best_similarity = None, None
        for entry in self._bucket(action):
            distance = _cosine_distance(embedding, entry["embedding"])
            similarity = 1.0 - distance
            if distance <= self.threshold and (best_similarity is None or similarity > best_similarity):
                best_answer, best_similarity = entry["answer"], similarity

        return best_answer is not None, best_answer, embedding, best_similarity

    def add(self, user_id: str, question: str, answer: str, embedding, action: str):
        """Store an answer scoped to this user's permissions (bucketed by action)."""
        if not has_access(user_id, action):
            return  # never store a denial-adjacent or unauthorized answer
        self._bucket(action).append({
            "question": question,
            "answer": answer,
            "embedding": np.asarray(embedding, dtype=float),
        })


def _run_pipeline(action: str, user_query: str) -> str:
    """Identical to Step 4 of secure_agentic_rag() — the actual retrieval/search call."""
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            return asyncio.run(route_function(user_query, action))
        return route_function(user_query, action)
    except Exception as exec_err:
        return f"Execution error: {exec_err}"


def secure_agentic_rag_cached(user_id: str, user_query: str, cache) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware semantic cache.

    Order: identity → route → permission → cache → pipeline.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
    """
    role = USERS.get(user_id)

    # 1. Identity — unknown users are rejected before any embedding, cache
    #    lookup, or LLM call ever runs.
    if role is None:
        return {"answer": f"🚫 Access denied: unknown user '{user_id}'.", "status": "DENIED", "role": None}

    # 2. Route — decide which source this query needs.
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        # Fail closed: we don't know what source this needs, so we can't
        # confirm permission — never cache, never run the pipeline.
        return {"answer": f"Routing error: {route_err}", "status": "DENIED", "role": role}

    action = decision.get("action")

    # 3. Permission — deny BEFORE the cache is ever touched, so a denial can
    #    never be served from (or written into) the cache.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        return {
            "answer": f"🚫 Access denied: your role ('{role}') does not have permission to query {source}.",
            "status": "DENIED",
            "role": role,
        }

    # Time-sensitive queries skip the cache entirely — always live, never stored.
    if is_time_sensitive(user_query):
        result = _run_pipeline(action, user_query)
        return {"answer": result, "status": "MISS", "role": role}

    # 4. Cache lookup — only ever reachable for an already-permitted (user, action).
    hit, cached_answer, embedding, similarity = cache.check(user_id, user_query, action)
    if hit:
        return {"answer": cached_answer, "status": "HIT", "role": role}

    # 5. Miss — run the real pipeline, then store, scoped to this action.
    result = _run_pipeline(action, user_query)
    cache.add(user_id, user_query, result, embedding, action)
    return {"answer": result, "status": "MISS", "role": role}

In [35]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


run_self_check()

✅ All checks passed — cache is fast and does not leak across roles.
